In [1]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch
import numpy as np

c:\Users\domen\Desktop\Agenti\Progetto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
encoder=SentenceTransformer("google/embeddinggemma-2")

Loading weights: 100%|██████████| 1376/1376 [00:04<00:00, 333.73it/s]


In [3]:
encoder.prompts

{'query': 'task: search result | query: ',
 'document': 'title: none | text: ',
 'BitextMining': 'task: search result | query: ',
 'Classification': 'task: classification | query: ',
 'Clustering': 'task: clustering | query: ',
 'CodeRetrieval': 'task: code retrieval | query: ',
 'Document': 'title: none | text: ',
 'FactChecking': 'task: fact checking | query: ',
 'InstructionRetrieval': 'task: code retrieval | query: ',
 'MultilabelClassification': 'task: classification | query: ',
 'PairClassification': 'task: sentence similarity | query: ',
 'QuestionAnswering': 'task: question answering | query: ',
 'Reranking': 'task: search result | query: ',
 'Retrieval': 'task: search result | query: ',
 'Retrieval-document': 'title: none | text: ',
 'Retrieval-query': 'task: search result | query: ',
 'STS': 'task: sentence similarity | query: ',
 'SearchQuery': 'task: search result | query: ',
 'SentenceSimilarity': 'task: sentence similarity | query: ',
 'Summarization': 'task: sentence sim

In [4]:
df_pdfs=pd.read_json("../data/processed/pdfs_pages_data.json",orient="records",lines=True)

In [5]:
df_pdfs_filter=df_pdfs[df_pdfs["n_chars"]>=100].copy()
df_pdfs_filter["embed_text"]="title: "+df_pdfs_filter["doc_name"].str.upper().replace("_"," ")+" | text: "+df_pdfs_filter["text"] 
df_pdfs_filter.reset_index(drop=True)#hai totlo delle righe quindi resetta gli indici
df_pdfs_filter.head()

,page_num,doc_name,n_chars,text,embed_text
0,0,3M_2015_10K,3880,Table of Contents\n \n \nlow\n \nUNITED STATES...,title: 3M_2015_10K | text: Table of Contents\n...
1,1,3M_2015_10K,1679,Table of Contents\n \n3M COMPANY\nFORM 10-K\nF...,title: 3M_2015_10K | text: Table of Contents\n...
2,2,3M_2015_10K,3862,Table of Contents\n \n3M COMPANY\nANNUAL REPOR...,title: 3M_2015_10K | text: Table of Contents\n...
3,3,3M_2015_10K,5266,Table of Contents\n \nglobal supplier of preci...,title: 3M_2015_10K | text: Table of Contents\n...
4,4,3M_2015_10K,5030,Table of Contents\n \nsurgical settings. Other...,title: 3M_2015_10K | text: Table of Contents\n...


In [6]:
torch.cuda.empty_cache()

In [7]:
embeddings_matrix=encoder.encode(df_pdfs_filter["embed_text"].tolist(),normalize_embeddings=True,batch_size=3,show_progress_bar=True)

Batches: 100%|██████████| 17749/17749 [1:11:27<00:00,  4.14it/s]


In [8]:
np.save("../data/processed/raw_embeddings.npy",embeddings_matrix)

In [10]:
embeddings_matrix.shape

(53247, 768)

In [ ]:
df_pdfs_filter.to_json("../data/processed/pdfs_filter.json",orient="records",lines=True,force_ascii=False)

In [9]:
embeddings_matrix[0:5]

array([[-7.8125000e-03,  1.7578125e-02, -3.4484863e-03, ...,
         3.5156250e-02, -4.2968750e-02, -2.5146484e-02],
       [ 5.5694580e-04,  2.2338867e-02,  3.3264160e-03, ...,
         3.9794922e-02, -2.9541016e-02, -1.7822266e-02],
       [-2.5390625e-02,  2.0629883e-02,  1.5747070e-02, ...,
         2.7587891e-02, -5.1269531e-02, -2.0385742e-02],
       [-1.0192871e-02,  3.5644531e-02,  2.9052734e-02, ...,
         1.4587402e-02, -2.9541016e-02, -3.3691406e-02],
       [ 2.7923584e-03,  2.7954102e-02, -6.3896179e-05, ...,
         3.9672852e-03, -1.4099121e-02, -9.8266602e-03]],
      shape=(5, 768), dtype=float32)

In [14]:
assert len(df_pdfs_filter) == embeddings_matrix.shape[0], \
    f"Righe diverse: DataFrame {len(df_pdfs_filter)}, matrice {embeddings_matrix.shape[0]}"

In [12]:
#carica su qudrant
df_pdfs_filter=pd.read_json("../data/processed/pdfs_filter.json",lines=True)
embeddings_matrix=np.load("../data/processed/raw_embeddings.npy")


In [4]:
client = QdrantClient(url="http://localhost:6333")

In [7]:
client.create_collection(
    collection_name="financebench_pages_embgemma2_title",
    vectors_config=models.VectorParams(
        size=encoder.get_embedding_dimension(),
        distance=models.Distance.COSINE
    ),
)

True

In [13]:
import uuid

def create_id(row):
    title=row["doc_name"]
    page_num=row["page_num"]
    return str(uuid.uuid5(uuid.NAMESPACE_URL,f"{title}_{page_num}"))
    
df_pdfs_filter["point_id"]=df_pdfs_filter.apply(create_id,axis=1)


In [14]:
df_pdfs_filter["point_id"].nunique()

53247

In [15]:
df_pdfs_filter.shape

(53247, 6)

In [16]:
client.upload_collection(
    collection_name="financebench_pages_embgemma2_title",
    vectors=embeddings_matrix,
    payload=df_pdfs_filter[["page_num","doc_name","n_chars","text"]].to_dict("records"),
    ids=df_pdfs_filter["point_id"].tolist(),
    batch_size=100
)

In [ ]:
question = "What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement."

query_vec = encoder.encode(question, prompt_name="QuestionAnswering", normalize_embeddings=True)

results = client.query_points(
    collection_name="financebench_pages_embgemma2_title",
    query=query_vec.tolist(),
    limit=5,
    with_payload=True,
).points

for p in results:
    print(p.payload["doc_name"], p.payload["page_num"], round(p.page_numpage_numscore, 4))

3M_2018_10K 48 0.8461
3M_2018_10K 45 0.8403
3M_2018_10K 46 0.8321
3M_2018_10K 38 0.8296
3M_2018_10K 59 0.8285
